In [ ]:
import os

# This walks through everything you attached and prints the exact file paths.
# Copy the paths you see here — you'll need them for the next steps.
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd

# This is your primary dataset (Recruitment Bias & Fairness)
PRIMARY_PATH = "/kaggle/input/datasets/vaishnavik212/checksum-datasets/archive/Dataset.csv"

df = pd.read_csv(PRIMARY_PATH)
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nFirst 5 rows:")
df.head()

In [ ]:
import numpy as np

np.random.seed(42)
n = len(df)
skill_col = "screening_score"

skill_z = (df[skill_col] - df[skill_col].mean()) / df[skill_col].std()

# --- college_tier: correlated with skill, not identical, not random ---
tier_noise = np.random.normal(0, 1, n)
tier_signal = 0.35 * skill_z + 0.65 * tier_noise
df["college_tier"] = pd.cut(tier_signal, bins=[-np.inf, -0.4, 0.4, np.inf], labels=[3, 2, 1]).astype(int)

# --- is_metro: same correlation logic ---
region_noise = np.random.normal(0, 1, n)
region_signal = 0.35 * skill_z + 0.65 * region_noise
df["is_metro"] = (region_signal > 0).astype(int)

# --- career_gap_months: mostly independent, realistic distribution ---
df["career_gap_months"] = np.random.exponential(scale=4, size=n).clip(0, 60).round().astype(int)

# --- noise-floor dummy fields (needed later for threshold calibration) ---
df["_noise_coin_flip"] = np.random.choice([0, 1], size=n)
df["_noise_shuffled_college_tier"] = np.random.permutation(df["college_tier"].values)

print("New columns added. Preview:")
df[["college_tier", "is_metro", "career_gap_months", "_noise_coin_flip"]].head()

In [ ]:
from scipy import stats

def partial_correlation(x, y, control):
    x_resid = x - np.polyval(np.polyfit(control, x, 1), control)
    y_resid = y - np.polyval(np.polyfit(control, y, 1), control)
    r, p = stats.pearsonr(x_resid, y_resid)
    return r, p

outcome_col = "shortlisted"
skill = df[skill_col].values.astype(float)
outcome = df[outcome_col].values.astype(float)

print("--- Proxy fields vs skill (should be moderate, e.g. 0.2-0.6) ---")
for col in ["college_tier", "is_metro"]:
    r, p = stats.pearsonr(df[col].values.astype(float), skill)
    print(f"  {col}: r={r:.3f}, p={p:.2e}")

print("\n--- Proxy fields vs outcome, controlling for skill (should be close to 0) ---")
for col in ["college_tier", "is_metro"]:
    r, p = partial_correlation(df[col].values.astype(float), outcome, skill)
    print(f"  {col}: partial_r={r:.3f}, p={p:.2e}")

print("\n--- Noise-floor fields (should show ~0 correlation with everything) ---")
for col in ["_noise_coin_flip", "_noise_shuffled_college_tier"]:
    r_skill, _ = stats.pearsonr(df[col].values.astype(float), skill)
    r_outcome, _ = stats.pearsonr(df[col].values.astype(float), outcome)
    print(f"  {col}: r_vs_skill={r_skill:.3f}, r_vs_outcome={r_outcome:.3f}")

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

feature_cols = [skill_col, "college_tier", "is_metro", "career_gap_months", "experience_years"]

X = df[feature_cols]
y = df[outcome_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, preds))
print("AUC:", roc_auc_score(y_test, probs))

In [ ]:
print("Outcome distribution (shortlisted):")
print(df["shortlisted"].value_counts(normalize=True))

print("\n--- Correlation of ALL original fields with outcome ---")
print(f"screening_score vs shortlisted: r={stats.pearsonr(df['screening_score'], df['shortlisted'])[0]:.3f}")
print(f"experience_years vs shortlisted: r={stats.pearsonr(df['experience_years'], df['shortlisted'])[0]:.3f}")
print(f"age vs shortlisted: r={stats.pearsonr(df['age'], df['shortlisted'])[0]:.3f}")

print("\n--- Mean shortlisted rate by gender ---")
print(df.groupby("gender")["shortlisted"].mean())

print("\n--- Mean shortlisted rate by education_level ---")
print(df.groupby("education_level")["shortlisted"].mean())

print("\n--- Mean screening_score by shortlisted outcome ---")
print(df.groupby("shortlisted")["screening_score"].mean())

In [ ]:
# The original 'shortlisted' column is noise (confirmed above) - reconstruct it
# from real skill signals, same honest logic as before: skill decides outcome,
# pedigree fields are NOT part of this formula.

np.random.seed(42)

label_signal = (
    0.8 * df["screening_score"]
    + 1.2 * df["experience_years"]
    + np.random.normal(0, 8, size=len(df))  # realistic hiring-judgment noise
)

outcome_score = (label_signal - label_signal.min()) / (label_signal.max() - label_signal.min()) * 100
df["shortlisted_v2"] = (outcome_score > np.percentile(outcome_score, 60)).astype(int)

# Sanity check: this new outcome SHOULD correlate with skill now
print("New outcome vs screening_score:", stats.pearsonr(df["screening_score"], df["shortlisted_v2"])[0])
print("New outcome vs experience_years:", stats.pearsonr(df["experience_years"], df["shortlisted_v2"])[0])
print("\nNew outcome distribution:")
print(df["shortlisted_v2"].value_counts(normalize=True))

In [ ]:
outcome_v2 = df["shortlisted_v2"].values.astype(float)

print("--- Proxy fields vs NEW outcome, controlling for skill (should be close to 0) ---")
for col in ["college_tier", "is_metro"]:
    r, p = partial_correlation(df[col].values.astype(float), outcome_v2, skill)
    print(f"  {col}: partial_r={r:.3f}, p={p:.2e}")

In [ ]:
import numpy as np
from scipy import stats

def partial_correlation_multi(x, y, controls_df):
    """Residualize x and y against MULTIPLE control variables (e.g. skill + experience)."""
    from numpy.linalg import lstsq
    C = np.column_stack([controls_df, np.ones(len(controls_df))])  # add intercept
    beta_x, _, _, _ = lstsq(C, x, rcond=None)
    beta_y, _, _, _ = lstsq(C, y, rcond=None)
    x_resid = x - C @ beta_x
    y_resid = y - C @ beta_y
    r, p = stats.pearsonr(x_resid, y_resid)
    return r, p

controls = df[["screening_score", "experience_years"]].astype(float)
outcome_v2 = df["shortlisted_v2"].values.astype(float)

print("--- Proxy fields vs NEW outcome, controlling for BOTH skill signals ---")
for col in ["college_tier", "is_metro"]:
    r, p = partial_correlation_multi(df[col].values.astype(float), outcome_v2, controls)
    print(f"  {col}: partial_r={r:.3f}, p={p:.2e}")

print("\n--- Sanity check: does college_tier correlate with experience_years? ---")
print("r =", stats.pearsonr(df["college_tier"], df["experience_years"])[0])

In [ ]:
# Group candidates into skill bands, then check if college_tier still predicts
# outcome WITHIN each band (i.e. among people with near-identical skill).
df["skill_band"] = pd.qcut(df["screening_score"] + 0.5 * df["experience_years"], q=5, labels=False)

print("--- Outcome rate by college_tier, WITHIN each skill band ---")
print(df.groupby(["skill_band", "college_tier"])["shortlisted_v2"].mean().unstack())

In [ ]:
feature_cols = ["screening_score", "college_tier", "is_metro", "career_gap_months", "experience_years"]

X = df[feature_cols]
y = df["shortlisted_v2"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, preds))
print("AUC:", roc_auc_score(y_test, probs))

In [ ]:
rng_tier = np.random.default_rng(1)
rng_region = np.random.default_rng(2)
rng_gap = np.random.default_rng(3)
rng_noise1 = np.random.default_rng(4)
rng_outcome = np.random.default_rng(5)

In [ ]:
outcome_noise = rng_outcome.normal(0, 8, size=len(df))

label_signal = (
    0.8 * df["screening_score"]
    + 1.2 * df["experience_years"]
    + outcome_noise
)

outcome_score = (label_signal - label_signal.min()) / (label_signal.max() - label_signal.min()) * 100
df["shortlisted_v2"] = (outcome_score > np.percentile(outcome_score, 60)).astype(int)

print("Regenerated. college_tier and shortlisted_v2 updated.")

In [ ]:
r, p = stats.pearsonr(df["college_tier"].values.astype(float), outcome_noise)
print(f"college_tier vs TRUE outcome noise: r={r:.3f}, p={p:.2e}")

In [ ]:
outcome_v2 = df["shortlisted_v2"].values.astype(float)
skill = df[skill_col].values.astype(float)

print("--- Proxy fields vs outcome, controlling for skill only ---")
for col in ["college_tier", "is_metro"]:
    r, p = partial_correlation(df[col].values.astype(float), outcome_v2, skill)
    print(f"  {col}: partial_r={r:.3f}, p={p:.2e}")

controls = df[["screening_score", "experience_years"]].astype(float)
print("\n--- Proxy fields vs outcome, controlling for BOTH skill signals ---")
for col in ["college_tier", "is_metro"]:
    r, p = partial_correlation_multi(df[col].values.astype(float), outcome_v2, controls)
    print(f"  {col}: partial_r={r:.3f}, p={p:.2e}")

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Mean absolute SHAP value per feature = how much it matters overall to the model
mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False)

print(importance_df)

# --- Skill vs Pedigree split ---
skill_features = ["screening_score", "experience_years"]
pedigree_features = ["college_tier", "is_metro", "career_gap_months"]

skill_weight = importance_df[importance_df["feature"].isin(skill_features)]["mean_abs_shap"].sum()
pedigree_weight = importance_df[importance_df["feature"].isin(pedigree_features)]["mean_abs_shap"].sum()
total = skill_weight + pedigree_weight

print(f"\nSkill reliance: {skill_weight/total*100:.1f}%")
print(f"Pedigree/proxy reliance: {pedigree_weight/total*100:.1f}%")

In [ ]:
# Pick a real candidate to demo with — someone with solid skill, currently Tier 3 (non-elite college)
demo_candidate = X_test[X_test["college_tier"] == 3].sort_values("screening_score", ascending=False).iloc[[0]]

print("=== Original candidate ===")
print(demo_candidate)
original_score = model.predict_proba(demo_candidate)[:, 1][0]
print(f"\nOriginal shortlist probability: {original_score:.3f} ({original_score*100:.1f}%)")

# --- Perturb: change ONLY college_tier, keep skill/experience/everything else identical ---
perturbed_candidate = demo_candidate.copy()
perturbed_candidate["college_tier"] = 1  # swap to Tier 1, nothing else changes

perturbed_score = model.predict_proba(perturbed_candidate)[:, 1][0]
print(f"\n=== Same candidate, ONLY college_tier changed from 3 -> 1 ===")
print(f"New shortlist probability: {perturbed_score:.3f} ({perturbed_score*100:.1f}%)")

delta = (perturbed_score - original_score) * 100
print(f"\n>>> Score changed by {delta:+.1f} percentage points <<<")
print(">>> Skill, experience, career gap, region — ALL UNCHANGED <<<")

In [ ]:
from scipy.stats import wilcoxon

# Take all candidates currently at Tier 3 (the "worst" pedigree label)
tier3_candidates = X_test[X_test["college_tier"] == 3].copy()

original_scores = model.predict_proba(tier3_candidates)[:, 1]

# Perturb: swap ONLY college_tier to 1, everything else stays identical
perturbed = tier3_candidates.copy()
perturbed["college_tier"] = 1
perturbed_scores = model.predict_proba(perturbed)[:, 1]

deltas = (perturbed_scores - original_scores) * 100  # in percentage points

print(f"Number of Tier 3 candidates tested: {len(tier3_candidates)}")
print(f"Average score change (Tier 3 -> Tier 1, skill unchanged): {deltas.mean():+.2f} pts")
print(f"Median score change: {np.median(deltas):+.2f} pts")
print(f"Std dev: {deltas.std():.2f}")
print(f"Min / Max change: {deltas.min():+.2f} / {deltas.max():+.2f}")

# Paired significance test — is this a real, consistent effect or just noise?
stat, p_value = wilcoxon(original_scores, perturbed_scores)
print(f"\nWilcoxon signed-rank test: statistic={stat:.1f}, p-value={p_value:.2e}")
print("Significant?" , "YES - this is a real, consistent effect" if p_value < 0.05 else "NO - could be noise")

# Compare against the noise-floor field, as a sanity check (per plan Section 5.8)
tier3_noise_test = X_test.copy()
tier3_noise_test["career_gap_months"] = tier3_noise_test["career_gap_months"] + 0  # baseline, no change
noise_perturbed = X_test.copy()
noise_perturbed["career_gap_months"] = 10  # arbitrary change to a less-central field, for comparison
noise_original_scores = model.predict_proba(X_test)[:, 1]
noise_perturbed_scores = model.predict_proba(noise_perturbed)[:, 1]
noise_deltas = (noise_perturbed_scores - noise_original_scores) * 100
print(f"\n[Reference] career_gap_months perturbation average change: {noise_deltas.mean():+.2f} pts (for comparison)")

In [ ]:
import json

# 1. Save the trained model
model.save_model("/kaggle/working/hiring_agent.json")

# 2. Save SHAP feature importances
importance_df.to_csv("/kaggle/working/shap_importances.csv", index=False)

# 3. Save the full results summary as one JSON - everything you'll need to hand back
results_summary = {
    "model_performance": {
        "accuracy": 0.9075,
        "auc": 0.98125
    },
    "shap_skill_vs_pedigree": {
        "skill_reliance_pct": 56.8,
        "pedigree_reliance_pct": 43.2,
        "feature_importances": importance_df.to_dict(orient="records")
    },
    "single_candidate_demo": {
        "screening_score": 88.02,
        "college_tier_before": 3,
        "college_tier_after": 1,
        "score_before_pct": 0.3,
        "score_after_pct": 47.3,
        "delta_pts": 47.0
    },
    "aggregate_perturbation_test": {
        "field_tested": "college_tier",
        "n_candidates": 127,
        "avg_delta_pts": 62.81,
        "median_delta_pts": 82.56,
        "std_dev": 39.24,
        "min_delta_pts": 0.43,
        "max_delta_pts": 99.61,
        "wilcoxon_p_value": 1.39e-22,
        "significant": True
    },
    "reference_control_field": {
        "field_tested": "career_gap_months",
        "avg_delta_pts": 2.09
    },
    "feature_cols": feature_cols,
    "skill_col": skill_col,
    "outcome_col": "shortlisted_v2"
}

with open("/kaggle/working/checksum_results_summary.json", "w") as f:
    json.dump(results_summary, f, indent=2)

# 4. Also save the full dataset with synthetic columns (useful for the Audit Agent backend)
df.to_csv("/kaggle/working/candidates_with_synthetic_columns.csv", index=False)

print("Saved 4 files to /kaggle/working/:")
print("  - hiring_agent.json (trained model)")
print("  - shap_importances.csv")
print("  - checksum_results_summary.json (everything, in one place)")
print("  - candidates_with_synthetic_columns.csv (full dataset)")